In [0]:
# Databricks notebook source
import time
import uuid
import pandas as pd
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
table_name = "ecommerce_analytics_dev.gold_layer.fact_sales"

def run_serverless_benchmark(iteration_name):
    """
    Runs the query with a unique 'salt' (UUID) to prevent 
    Serverless from returning a cached result.
    """
    # A unique comment ensures the query plan is unique to Spark's cache
    query_salt = str(uuid.uuid4())
    
    test_query = f"""
        /* {query_salt} */
        SELECT product_id, SUM(price) as total_revenue
        FROM {table_name}
        WHERE event_time BETWEEN '2019-10-01' AND '2019-10-07'
        GROUP BY product_id
        ORDER BY total_revenue DESC
        LIMIT 10
    """
    
    print(f"🚀 Running {iteration_name}...")
    start_time = time.time()
    spark.sql(test_query).collect()
    duration = round(time.time() - start_time, 2)
    print(f"⏱️ {iteration_name} Duration: {duration} seconds")
    return duration

# --- 1. BASELINE RUN ---
# Run it twice to see if Serverless has already warmed up, 
# then take the average or the first run as the 'cold' baseline.
baseline_duration = run_serverless_benchmark("Baseline (Non-Optimized)")

# --- 2. APPLY OPTIMIZATIONS ---
print(f"🛠️ Applying OPTIMIZE and ZORDER to {table_name}...")
# This physically reorganizes the Parquet files on disk
spark.sql(f"OPTIMIZE {table_name} ZORDER BY (event_time)")

# --- 3. OPTIMIZED RUN ---
# We use the same 'salt' logic to ensure we are testing the DISK speed, not the CACHE speed.
optimized_duration = run_serverless_benchmark("Optimized (ZORDER)")

# --- 4. CALCULATE IMPROVEMENT ---
improvement = ((baseline_duration - optimized_duration) / baseline_duration) * 100
print(f"\n✅ Performance Improvement: {improvement:.2f}%")

# --- 5. VISUAL PROOF ---
results_df = pd.DataFrame({
    'Metric': ['Baseline', 'Optimized (ZORDER)'],
    'Seconds': [baseline_duration, optimized_duration]
})

results_df.plot(kind='bar', x='Metric', y='Seconds', color=['#E74C3C', '#2ECC71'], legend=False)
plt.title("Serverless Query Latency: 110M Rows")
plt.ylabel("Time (Seconds)")
plt.xticks(rotation=0)
plt.show()

In [0]:
%sql
-- Proof of physical Z-Ordering
DESCRIBE DETAIL ecommerce_analytics_dev.gold_layer.fact_sales;